# CSV / Excel / Parquet / SQLite

داده‌های این نمونه ساختگی هستند؛ قیمت واقعی بازار نیستند.

## سلول ۱ — ورود کتابخانه‌ها

In [ ]:
from pathlib import Path
import tempfile
import pandas as pd
import pydara
from pydara import tse
from pydara.tse.samples import sample_client
from pydara.tse.exceptions import OptionalDependencyError
print(pydara.__version__)

## سلول ۲ — تعریف توابع

In [ ]:
def roundtrip(frame, path):
    receipt = tse.save(frame, path)
    restored = tse.load(path)
    pd.testing.assert_frame_equal(frame, restored)
    assert frame.attrs == restored.attrs
    return {"format": path.suffix, "status": "passed", "rows": len(restored)}

## سلول ۳ — تنظیمات

In [ ]:
SYMBOL = 'فولاد'
OUTPUT = Path(tempfile.mkdtemp(prefix="pydara-part3-files-"))
FORMATS = ["csv", "xlsx", "parquet", "sqlite"]
print("Output:", OUTPUT)

## سلول ۴ — اجرا

In [ ]:
with sample_client() as client:
    df = client.get_history(SYMBOL, include_jdate=True)
results = []
for extension in FORMATS:
    try:
        results.append(roundtrip(df, OUTPUT / ("synthetic_prices." + extension)))
    except OptionalDependencyError as exc:
        results.append({"format": extension, "status": "not_tested_missing_engine", "message": str(exc)})

## سلول ۵ — گزارش نتیجه

In [ ]:
display(pd.DataFrame(results))
display(df[["date", "ins_code", "closing_price", "volume"]])
print("Keep each CSV together with its .pydara.json sidecar.")